# Feature Engineering

In [3]:
import warnings
warnings.filterwarnings('ignore')

import re
import math
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import Counter
from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
import pickle

## Data Ingestion

### Hugging Face token (optional)

For higher rate limits and faster downloads, use a [Hugging Face token](https://huggingface.co/settings/tokens). **Do not commit your token.**

- **Option A:** Set the `HF_TOKEN` environment variable before starting the notebook (e.g. `export HF_TOKEN=hf_xxxx` in terminal).
- **Option B:** In the cell below, uncomment and paste your token (session-only).

In [ ]:
import os
#os.environ["HF_TOKEN"] = ""  # uncomment and paste your token
#HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")

In [10]:
dataset = load_dataset("ms_marco", "v2.1", split="validation", token=HF_TOKEN)
marco_df = dataset.to_pandas()
marco_df.head()

Generating test split: 100%|██████████| 101092/101092 [00:01<00:00, 67801.92 examples/s]


,answers,passages,query,query_id,query_type,wellFormedAnswers
0,[A corporation is a company or group of people...,"{'is_selected': [0, 0, 0, 0, 0, 1, 0, 0, 0, 0]...",. what is a corporation?,1102432,DESCRIPTION,[]
1,[Rachel Carson writes The Obligation to Endure...,"{'is_selected': [0, 0, 0, 0, 1, 1, 0, 0, 0, 0]...",why did rachel carson write an obligation to e...,1102431,DESCRIPTION,[]
2,[No Answer Present.],"{'is_selected': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]...",why did the progressive movement fail to advan...,1102421,DESCRIPTION,[]
3,[No Answer Present.],"{'is_selected': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]...",why do police need to understand what the fore...,1102315,DESCRIPTION,[]
4,[No Answer Present.],"{'is_selected': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]...",do owls eat in the day,1101280,NUMERIC,[]


| Column | Type | Notes |
| :--- | :--- | :--- |
| query | string | The search query |
| passages | dict | Contains passage_text (list) and is_selected (list) |
| query_id | int | Unique query identifier |

In [36]:
marco_df.shape

(101093, 6)

Current size might be a bit too large for development purposes. Would be sub-sampling to 10,000 rows using random_state of 42 to ensure reproducibility

In [37]:
marco_df_samp = marco_df.sample(n=10000, random_state=42).reset_index(drop=True)

## Data Pre-processing

### Explode Rows

In [38]:
marco_df.columns

Index(['answers', 'passages', 'query', 'query_id', 'query_type',
       'wellFormedAnswers'],
      dtype='str')

In [39]:
marco_df_exp = marco_df_samp.copy()
marco_df_exp['passage_text'] = marco_df_samp['passages'].apply(lambda x: x['passage_text'])
marco_df_exp['is_selected'] = marco_df_samp['passages'].apply(lambda x: x['is_selected'])

marco_df_exp = marco_df_exp.explode(['passage_text', 'is_selected']).reset_index(drop=True)
marco_df_exp['label'] = marco_df_exp['is_selected'].astype(int)
print(marco_df_exp.shape) 

(99840, 9)


In [40]:
valid_query_ids = marco_df_exp.groupby('query_id')['is_selected'].sum()
valid_query_ids = valid_query_ids[valid_query_ids > 0].index
marco_df_exp = marco_df_exp[marco_df_exp['query_id'].isin(valid_query_ids)]

print(f"Remaining rows: {marco_df_exp.shape[0]:,}")
print(f"Unique queries: {marco_df_exp['query_id'].nunique():,}")

Remaining rows: 54,281
Unique queries: 5,435


## Feature Engineering

#### Lexical Matching Features
*"Do the query words actually appear in the passage?"*

| Feature | Why It Matters |
| :--- | :--- |
| BM25 score | The gold standard sparse retrieval signal, what Bing/Google used before neural models |
| TF-IDF cosine similarity | Measures vocabulary overlap between query and passage |
| Query term coverage | % of query words found in the passage |
| Exact match flag | Binary - does the full query appear verbatim in the passage? |

#### Semantic Features
*"Do they mean the same thing, even with different words?"*
| Feature | Why It Matters |
| :--- | :--- |
| TF-IDF vector similarity | Captures related vocabulary beyond exact matches |
| Query-passage Jaccard similarity | Simple word overlap ratio |

We may substitute TF-IDF for BERT depending on computing power

#### Statistical / Length Features
*"What does the structure of the passage tell us?"*

| Feature | Why It Matters |
| :--- | :--- |
| Passage length (word count) | Longer passages may dilute relevance |
| Query length (word count) | Longer queries are more specific |
| Length ratio | query length / passage length |
| Passage position | Earlier passages in the original list tend to be more relevant |

#### Label 

| Column | Role |
| :--- | :--- |
| is_selected | Our binary target where 1 = relevant, 0 = not relevant |




#### Query Length, Passage Length, Length Ratio

In [41]:
marco_df_exp['query_length'] = marco_df_exp['query'].str.split().str.len()
marco_df_exp['passage_length'] = marco_df_exp['passage_text'].str.split().str.len()
marco_df_exp['length_ratio'] = marco_df_exp['query_length'] / (marco_df_exp['passage_length'] + 1)
print(marco_df_exp[['query_length', 'passage_length', 'length_ratio']].describe())

       query_length  passage_length  length_ratio
count  54281.000000    54281.000000  54281.000000
mean       5.907574       51.209889      0.126205
std        2.540894       18.997756      0.074595
min        2.000000        1.000000      0.014286
25%        4.000000       40.000000      0.078431
50%        6.000000       48.000000      0.111111
75%        7.000000       57.000000      0.153846
max       30.000000      196.000000      2.500000


#### Exact match flag, Query term coverage

In [42]:
def exact_match(query, passage):
    return int(str(query).lower() in str(passage).lower())

def query_term_coverage(query, passage):
    query_words = set(str(query).lower().split())
    passage_words = set(str(passage).lower().split())
    if len(query_words) == 0:
        return 0.0
    return len(query_words & passage_words) / len(query_words)

In [43]:
marco_df_exp['exact_match'] = marco_df_exp.apply(
    lambda row: exact_match(row['query'], row['passage_text']), axis=1
)

marco_df_exp['query_term_coverage'] = marco_df_exp.apply(
    lambda row: query_term_coverage(row['query'], row['passage_text']), axis=1
)

In [44]:
print(marco_df_exp[['exact_match', 'query_term_coverage']].describe())

        exact_match  query_term_coverage
count  54281.000000         54281.000000
mean       0.010464             0.441115
std        0.101758             0.240172
min        0.000000             0.000000
25%        0.000000             0.272727
50%        0.000000             0.428571
75%        0.000000             0.600000
max        1.000000             1.000000


#### Jaccard Similarity

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

In [45]:
def jaccard_similarity(query, passage):
    query_words = set(str(query).lower().split())
    passage_words = set(str(passage).lower().split())
    if len(query_words | passage_words) == 0:
        return 0.0
    return len(query_words & passage_words) / len(query_words | passage_words)

In [46]:
marco_df_exp['jaccard_similarity'] = marco_df_exp.apply(
    lambda row: jaccard_similarity(row['query'], row['passage_text']), axis=1
)

In [47]:
print(marco_df_exp[['jaccard_similarity']].describe())

       jaccard_similarity
count        54281.000000
mean             0.063802
std              0.046048
min              0.000000
25%              0.031250
50%              0.056818
75%              0.085714
max              1.000000


#### TF-IDF Similarity

- TF-IDF Weighting
$$w_{t,d} = \text{tf}_{t,d} \times \log\left(\frac{N}{\text{df}_t}\right)$$

- Cosine Similarity (The "Similarity" Part)
$$\text{similarity}(A, B) = \frac{A \cdot B}{\|A\| \|B\|} = \frac{\sum_{i=1}^{n} A_i B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \sqrt{\sum_{i=1}^{n} B_i^2}}$$

In [48]:
all_text = pd.concat([marco_df_exp['query'], marco_df_exp['passage_text']], ignore_index=True)
vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True)
vectorizer.fit(all_text)

BATCH_SIZE = 5000
tfidf_scores = []

for i in tqdm(range(0, len(marco_df_exp), BATCH_SIZE), desc="TF-IDF batches"):
    batch = marco_df_exp.iloc[i:i+BATCH_SIZE]
    query_vecs = vectorizer.transform(batch['query'])
    passage_vecs = vectorizer.transform(batch['passage_text'])
    scores = cosine_similarity(query_vecs, passage_vecs).diagonal()
    tfidf_scores.extend(scores)

marco_df_exp['tfidf_cosine_sim'] = tfidf_scores


TF-IDF batches: 100%|██████████| 11/11 [00:05<00:00,  2.08it/s]


In [49]:
marco_df_exp['tfidf_cosine_sim'].describe()

count    54281.000000
mean         0.194105
std          0.139623
min          0.000000
25%          0.088628
50%          0.174015
75%          0.279113
max          1.000000
Name: tfidf_cosine_sim, dtype: float64

In [ ]:
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

#### BM25

BM25 is a bag-of-words retrieval function that ranks a set of documents based on the query terms appearing in each document, regardless of their proximity within the document. It is a family of scoring functions with slightly different components and parameters. One of the most prominent instantiations of the function is as follows.

$$\text{score}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$


- $f(q_i, D)$: Number of times the query term $q_i$ occurs in document $D$
- $|D|$: Length of document $D$ (in words)
- $\text{avgdl}$: Average document length in the collection
- $k_1$: Term frequency saturation parameter  
  - Typically $k_1 \in [1.2, 2.0]$
- $b$: Length normalization parameter  
  - Commonly set to $b = 0.75$


The inverse document frequency weight for a query term \( q_i \) is usually computed as:
$$\text{IDF}(q_i) = \ln \left( \frac{N - n(q_i) + 0.5}{n(q_i) + 0.5} + 1 \right)$$

where:
- $(N)$: Total number of documents in the collection
- $(n(q_i))$: Number of documents containing term $(q_i)$

In [50]:

print("Tokenizing passages...")
tokenized_passages = [str(p).lower().split() for p in marco_df_exp['passage_text']]
bm25 = BM25Okapi(tokenized_passages)
print(f"BM25 index built over {len(tokenized_passages):,} passages")

# Get unique queries
unique_queries = marco_df_exp[['query_id', 'query']].drop_duplicates()
print(f"Scoring {len(unique_queries):,} unique queries...")

# Score each unique query once
bm25_score_map = {}
for _, row in tqdm(unique_queries.iterrows(), total=len(unique_queries), desc="BM25 scoring"):
    query_tokens = str(row['query']).lower().split()
    bm25_score_map[row['query_id']] = bm25.get_scores(query_tokens)

# Map scores back
marco_df_exp = marco_df_exp.reset_index(drop=True)
marco_df_exp['bm25_score'] = [
    bm25_score_map[row['query_id']][idx]
    for idx, row in marco_df_exp.iterrows()
]


Tokenizing passages...
BM25 index built over 54,281 passages
Scoring 5,435 unique queries...


BM25 scoring: 100%|██████████| 5435/5435 [08:54<00:00, 10.16it/s]


In [ ]:
# with open('bm25_scores.pkl', 'wb') as f:
#     pickle.dump(marco_df_exp['bm25_score'].tolist(), f)

# with open('bm25_index.pkl', 'wb') as f:
#     pickle.dump(bm25, f)

# print(f"BM25 done — saved to bm25_scores.pkl and bm25_index.pkl")
# print(marco_df_exp['bm25_score'].describe().round(3))

BM25 done — saved to bm25_scores.pkl and bm25_index.pkl
count    54281.000
mean        15.491
std         10.302
min          0.000
25%          8.263
50%         14.647
75%         21.594
max        119.100
Name: bm25_score, dtype: float64


#### Passage Position

In [51]:
marco_df_exp['passage_position'] = marco_df_exp.groupby('query_id').cumcount()

#### IDF-weighted overlap and first query term position

- **idf_weighted_overlap**: Sum of IDF(term) over query terms that appear in the passage. IDF from passage corpus: $\\text{IDF}(t) = \\ln((N - n(t) + 0.5)/(n(t) + 0.5) + 1)$.
- **first_query_term_position_norm**: Word index of first occurrence of any query term in the passage, normalized by passage length (0 = start, 1 = end or no match).

In [52]:
# Compute IDF from the passage corpus (same tokenization as BM25)
N = len(tokenized_passages)
doc_freq = Counter()
for tokens in tokenized_passages:
    for term in set(tokens):
        doc_freq[term] += 1
idf = {
    term: math.log((N - doc_freq[term] + 0.5) / (doc_freq[term] + 0.5) + 1)
    for term in doc_freq
}
print(f"IDF computed over {N:,} passages; {len(idf):,} unique terms")

def idf_weighted_overlap(row, tokenized_passages, idf_dict):
    """Sum of IDF(term) over query terms that appear in the passage."""
    query_tokens = str(row["query"]).lower().split()
    passage_tokens = tokenized_passages[row.name]
    passage_set = set(passage_tokens)
    matched = set(query_tokens) & passage_set
    return sum(idf_dict.get(t, 0.0) for t in matched)


def first_query_term_position_norm(row, tokenized_passages):
    """Word index of first occurrence of any query term in passage, normalized by passage length."""
    query_tokens = str(row["query"]).lower().split()
    passage_tokens = tokenized_passages[row.name]
    first_pos = len(passage_tokens)
    for q in query_tokens:
        if q in passage_tokens:
            first_pos = min(first_pos, passage_tokens.index(q))
    return first_pos / len(passage_tokens) if passage_tokens else 0.0


marco_df_exp["idf_weighted_overlap"] = marco_df_exp.apply(
    lambda row: idf_weighted_overlap(row, tokenized_passages, idf),
    axis=1,
)
marco_df_exp["first_query_term_position_norm"] = marco_df_exp.apply(
    lambda row: first_query_term_position_norm(row, tokenized_passages),
    axis=1,
)
print(marco_df_exp[["idf_weighted_overlap", "first_query_term_position_norm"]].describe().round(4))

IDF computed over 54,281 passages; 180,981 unique terms
       idf_weighted_overlap  first_query_term_position_norm
count            54281.0000                      54281.0000
mean                10.2636                          0.2093
std                  7.1034                          0.2994
min                  0.0000                          0.0000
25%                  5.5165                          0.0185
50%                  9.4245                          0.0769
75%                 14.7844                          0.2442
max                 82.7804                          1.0000


### Final Check of Features

In [53]:
feature_cols = [
    'query_length', 'passage_length', 'length_ratio',
    'exact_match', 'query_term_coverage', 'jaccard_similarity',
    'tfidf_cosine_sim', 'bm25_score', 'passage_position',
    'idf_weighted_overlap', 'first_query_term_position_norm',
]

In [54]:
print("Feature Summary:")
print(marco_df_exp[feature_cols].describe().round(3))

Feature Summary:
       query_length  passage_length  length_ratio  exact_match  \
count     54281.000       54281.000     54281.000    54281.000   
mean          5.908          51.210         0.126        0.010   
std           2.541          18.998         0.075        0.102   
min           2.000           1.000         0.014        0.000   
25%           4.000          40.000         0.078        0.000   
50%           6.000          48.000         0.111        0.000   
75%           7.000          57.000         0.154        0.000   
max          30.000         196.000         2.500        1.000   

       query_term_coverage  jaccard_similarity  tfidf_cosine_sim  bm25_score  \
count            54281.000           54281.000         54281.000   54281.000   
mean                 0.441               0.064             0.194      15.491   
std                  0.240               0.046             0.140      10.302   
min                  0.000               0.000             0.000    

#### IDF-weighted overlap and first query term position

- **idf_weighted_overlap**: Sum of IDF(term) over query terms that appear in the passage (rare matching terms contribute more). IDF is computed from the passage corpus: $\\text{IDF}(t) = \\ln\\left(\\frac{N - n(t) + 0.5}{n(t) + 0.5} + 1\\right)$.
- **first_query_term_position_norm**: Word index of the first occurrence of any query term in the passage, normalized by passage length (0 = match at start, 1 = at end or no match). Captures whether the passage leads with query-related content.

In [55]:
# Compute IDF from the passage corpus (same tokenization as BM25)
N = len(tokenized_passages)
doc_freq = Counter()
for tokens in tokenized_passages:
    for term in set(tokens):
        doc_freq[term] += 1
idf = {
    term: math.log((N - doc_freq[term] + 0.5) / (doc_freq[term] + 0.5) + 1)
    for term in doc_freq
}
print(f"IDF computed over {N:,} passages; {len(idf):,} unique terms")

def idf_weighted_overlap(row, tokenized_passages, idf_dict):
    """Sum of IDF(term) over query terms that appear in the passage."""
    query_tokens = str(row["query"]).lower().split()
    passage_tokens = tokenized_passages[row.name]
    passage_set = set(passage_tokens)
    matched = set(query_tokens) & passage_set
    return sum(idf_dict.get(t, 0.0) for t in matched)


def first_query_term_position_norm(row, tokenized_passages):
    """Word index of first occurrence of any query term in passage, normalized by passage length."""
    query_tokens = str(row["query"]).lower().split()
    passage_tokens = tokenized_passages[row.name]
    first_pos = len(passage_tokens)
    for q in query_tokens:
        if q in passage_tokens:
            first_pos = min(first_pos, passage_tokens.index(q))
    return first_pos / len(passage_tokens) if passage_tokens else 0.0


marco_df_exp["idf_weighted_overlap"] = marco_df_exp.apply(
    lambda row: idf_weighted_overlap(row, tokenized_passages, idf),
    axis=1,
)
marco_df_exp["first_query_term_position_norm"] = marco_df_exp.apply(
    lambda row: first_query_term_position_norm(row, tokenized_passages),
    axis=1,
)
print(marco_df_exp[["idf_weighted_overlap", "first_query_term_position_norm"]].describe().round(4))

IDF computed over 54,281 passages; 180,981 unique terms
       idf_weighted_overlap  first_query_term_position_norm
count            54281.0000                      54281.0000
mean                10.2636                          0.2093
std                  7.1034                          0.2994
min                  0.0000                          0.0000
25%                  5.5165                          0.0185
50%                  9.4245                          0.0769
75%                 14.7844                          0.2442
max                 82.7804                          1.0000


In [56]:
print(f"\nShape: {marco_df_exp.shape}")


Shape: (54281, 20)


In [57]:
print(f"Missing values:\n{marco_df_exp[feature_cols].isnull().sum()}")

Missing values:
query_length                      0
passage_length                    0
length_ratio                      0
exact_match                       0
query_term_coverage               0
jaccard_similarity                0
tfidf_cosine_sim                  0
bm25_score                        0
passage_position                  0
idf_weighted_overlap              0
first_query_term_position_norm    0
dtype: int64


In [58]:
# Save final dataframe
marco_df_exp.to_pickle('marco_df_new_feats.pkl')
print("All features saved to marco_df_new_feats.pkl")

All features saved to marco_df_new_feats.pkl


### Model Dev Next Steps - If you don't want to run feature engineering

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# import pickle
# import pandas as pd

# # Load the finished dataframe — skip all feature engineering
# df_exploded = pd.read_pickle('/content/drive/MyDrive/YOUR_FOLDER/df_features.pkl')

# # Load vectorizer and BM25 index if needed later
# with open('/content/drive/MyDrive/YOUR_FOLDER/tfidf_vectorizer.pkl', 'rb') as f:
#     vectorizer = pickle.load(f)

# with open('/content/drive/MyDrive/YOUR_FOLDER/bm25_index.pkl', 'rb') as f:
#     bm25 = pickle.load(f)

# print(df_exploded.shape)
# print(df_exploded.columns.tolist())